<a href="https://colab.research.google.com/github/EliVil2/RUTAS-IO/blob/mismo-intento-2-con-links-de-google-maps/Intento_2_IO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install openrouteservice folium geopy ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [6]:
# === IMPORTACIONES ===
import folium
import openrouteservice
from openrouteservice import convert
import numpy as np
from sklearn.cluster import KMeans

# === CONFIGURACIÓN ===
ORS_API_KEY = '5b3ce3597851110001cf6248b75b48749e4e41a38cdaea0746c5ae37'
client = openrouteservice.Client(key=ORS_API_KEY)

# === DATOS DE ENTRADA ===
locations = [
    (10.983724, -74.789957),  # origen Universidad sede centro
    (10.9700532, -74.80195069999999), (10.9525577, -74.80394919999999), (10.975318, -74.808005),
    (10.9066255, -74.7856385), (10.9724651, -74.80604559999999), (11.017676, -74.809716),
    (10.944379, -74.803011), (10.9124556, -74.7826908), (10.9606144, -74.8028657),
    (10.9230761, -74.8008353), (10.9628562, -74.83497729999999), (10.940902, -74.771705),
    (10.9942585, -74.8122456), (10.963991, -74.81723699999999), (11.027218, -74.869495),
    (10.9615811, -74.83029429999999), (10.979663, -74.80364999999999), (10.9992743, -74.7924869),
    (10.9625489, -74.83082399999999), (10.985877, -74.83556), (11.024694, -74.86955499999999),
    (10.932394, -74.766357), (11.023623, -74.80664), (10.9030214, -74.7919232), (11.02425,-74.86800),
    (10.9558219, -74.8218484), (10.9599135, -74.8071882), (10.894661, -74.885339),
    (10.9590029, -74.796548), (10.7465427, -74.7565432), (11.0218788, -74.8705983),
]
demands = [0, 107, 112, 95, 84, 72, 107, 52, 110, 49, 67, 75, 107, 111, 107, 80, 87, 94, 78, 66, 30,
           70, 69, 94, 70, 114, 64, 46, 76, 33, 60, 69]
vehicle_capacity = 500

# === FUNCIONES AUXILIARES ===
def get_route(coords):
    try:
        route = client.directions(coords, profile='driving-car', format='geojson')
        distance = route['features'][0]['properties']['segments'][0]['distance']
        geometry = route['features'][0]['geometry']
        return distance, geometry
    except Exception as e:
        print("Error con ruta:", coords)
        return float('inf'), None

def generar_enlace_google_maps(locations, route):
    base_url = "https://www.google.com/maps/dir/"
    waypoints = [f"{locations[i][0]},{locations[i][1]}" for i in route]
    return base_url + "/".join(waypoints)

# === AGRUPACIÓN (CLUSTERING) POR DEMANDA Y CAPACIDAD ===
num_vehicles = int(np.ceil(sum(demands[1:]) / vehicle_capacity))
coords_np = np.array(locations[1:])
kmeans = KMeans(n_clusters=num_vehicles, random_state=42).fit(coords_np)

# Asignar puntos a clusters
clusters = [[] for _ in range(num_vehicles)]
for idx, label in enumerate(kmeans.labels_):
    clusters[label].append(idx + 1)

# Redistribuir puntos si un clúster supera la capacidad
def balancear_clusters(clusters, demands, capacidad):
    nuevos_clusters = [[] for _ in range(len(clusters))]
    pesos_actuales = [0] * len(clusters)

    puntos_disponibles = sorted([(i, demands[i]) for c in clusters for i in c], key=lambda x: -x[1])

    for idx, demanda in puntos_disponibles:
        for i in range(len(nuevos_clusters)):
            if pesos_actuales[i] + demanda <= capacidad:
                nuevos_clusters[i].append(idx)
                pesos_actuales[i] += demanda
                break
    return nuevos_clusters

clusters = balancear_clusters(clusters, demands, vehicle_capacity)

# === VECINO MÁS CERCANO ===
def nearest_neighbor_route(cluster):
    unvisited = set(cluster)
    current = 0
    route = [0]
    while unvisited:
        next_stop = min(unvisited, key=lambda x: np.linalg.norm(np.array(locations[current]) - np.array(locations[x])))
        route.append(next_stop)
        unvisited.remove(next_stop)
        current = next_stop
    return route

# === MAPA Y RESULTADOS ===
all_routes = []
total_distance = 0
m = folium.Map(location=locations[0], zoom_start=12)
colors = ['blue', 'green', 'red', 'purple', 'orange', 'darkred', 'cadetblue']

for i, cluster in enumerate(clusters):
    route = nearest_neighbor_route(cluster)
    route_coords = [locations[i] for i in route]
    total_weight = sum(demands[i] for i in route[1:])
    distance = 0

    for j in range(len(route_coords) - 1):
        segment = [tuple(reversed(route_coords[j])), tuple(reversed(route_coords[j + 1]))]
        dist, geometry = get_route(segment)
        distance += dist
        if geometry:
            folium.GeoJson(geometry,
                           name=f"Ruta Vehículo {i + 1}",
                           style_function=lambda x, color=colors[i % len(colors)]: {'color': color, 'weight': 4}
                           ).add_to(m)

    total_distance += distance
    enlace = generar_enlace_google_maps(locations, route)
    all_routes.append((i + 1, route, total_weight, distance / 1000, enlace))

# === MOSTRAR RESULTADOS ===
for veh_id, route, weight, dist_km, enlace in all_routes:
    print(f"Vehículo {veh_id}: Ruta: {route} | Peso total: {weight} kg | Distancia: {dist_km:.2f} km")
    print(f"Google Maps: {enlace}\n")

m.save("rutas_vehiculos.html")
print("Mapa guardado como rutas_vehiculos.html")


Vehículo 1: Ruta: [0, 13, 2, 7, 8, 25] | Peso total: 499 kg | Distancia: 40.33 km
Google Maps: https://www.google.com/maps/dir/10.983724,-74.789957/10.9942585,-74.8122456/10.9525577,-74.80394919999999/10.944379,-74.803011/10.9124556,-74.7826908/11.02425,-74.868

Vehículo 2: Ruta: [0, 1, 5, 14, 12, 6] | Peso total: 500 kg | Distancia: 25.81 km
Google Maps: https://www.google.com/maps/dir/10.983724,-74.789957/10.9700532,-74.80195069999999/10.9724651,-74.80604559999999/10.963991,-74.81723699999999/10.940902,-74.771705/11.017676,-74.809716

Vehículo 3: Ruta: [0, 17, 3, 27, 16, 23, 4] | Peso total: 500 kg | Distancia: 34.67 km
Google Maps: https://www.google.com/maps/dir/10.983724,-74.789957/10.979663,-74.80364999999999/10.975318,-74.808005/10.9599135,-74.8071882/10.9615811,-74.83029429999999/11.023623,-74.80664/10.9066255,-74.7856385

Vehículo 4: Ruta: [0, 18, 9, 11, 21, 15, 28, 24] | Peso total: 498 kg | Distancia: 63.39 km
Google Maps: https://www.google.com/maps/dir/10.983724,-74.789957